In [ ]:
# first run manually: scripts/streaming.py
# TODO integrate in the bench

In [ ]:
from pathlib import Path
import pickle
from collections import defaultdict, Counter
from typing import Any

import pandas as pd
import matplotlib.pyplot as plt

from asr_eval.streaming.evaluation import RecordingStreamingEvaluation
from asr_eval.streaming.plots import partial_alignments_plot, streaming_error_vs_latency_histogram, latency_plot, show_last_alignments, visualize_history

In [ ]:
evals: dict[str, dict[int, RecordingStreamingEvaluation]] = defaultdict(dict)

for path in Path('outputs/streaming_evals').glob('*/common-voice-17.0/*.pkl'):
    # print(path)
    model_name = path.parts[-3]
    sample_idx = int(path.stem)
    try:
        evals[model_name][sample_idx] = pickle.loads(path.read_bytes())
    except EOFError:
        pass # print('EOFError')

In [ ]:
sample_indices: set[int] = set(next(iter(evals.values())))
sample_indices_clarification: set[int] = set(evals['psz-dev-branch-commit-hash-be884dcf-clarification'])
for model_name, model_evals in evals.items():
    print(model_name)
    if model_name != 'psz-dev-branch-commit-hash-be884dcf-clarification':
        sample_indices &= set(model_evals)
    sample_indices_clarification &= set(model_evals)

In [ ]:
fig, axs = plt.subplots(ncols=3, nrows=2, figsize=(15, 6)) # type:ignore
for model_idx, (model_name, model_evals) in enumerate(evals.items()):
    ax = axs.flat[model_idx] # type: ignore
    ax.set_title(model_name) # type: ignore
    latency_plot(list(model_evals.values()), ax=ax) # type: ignore
plt.tight_layout() # type: ignore

In [ ]:
for indices_set_name, indices_set in (
    ('clarification (53 samples)', sample_indices_clarification),
    ('all (493 samples)', sample_indices),
):
    print(indices_set_name)
    rows: list[dict[str, Any]] = []
    for model_name in evals:
        if model_name == 'psz-dev-branch-commit-hash-be884dcf-clarification' and indices_set_name != 'clarification (53 samples)':
            continue
        samples = [
            eval for sample_idx, eval in evals[model_name].items()
            if sample_idx in indices_set
        ]
        err_counter = Counter([
            err.status
            for eval in samples
            for err in eval.partial_alignments[-1].get_error_positions()
        ])
        n_errors = (
            err_counter['insertion']
            + err_counter['replacement']
            + err_counter['deletion']
            + err_counter['not_yet']
        )
        rows.append({
            # 'indices_set_name': indices_set_name,
            'model_name': model_name,
            **err_counter,
            'errors': n_errors,
        })
        # print(model_name, sum([v for k, v in err_counter.items() if k != 'insertion']), err_counter)
        
    df = pd.DataFrame(rows)
    df['not_yet'] = df['not_yet'].fillna(0).astype(int) # type: ignore
    display(df)

In [ ]:
fig, axs = plt.subplots(ncols=3, nrows=2, figsize=(15, 6)) # type:ignore
for model_idx, (model_name, model_evals) in enumerate(evals.items()):
    ax = axs.flat[model_idx] # type: ignore
    ax.set_title(model_name) # type: ignore
    streaming_error_vs_latency_histogram(list(model_evals.values()), ax=ax) # type: ignore
plt.tight_layout() # type: ignore

In [ ]:
fig, axs = plt.subplots(ncols=3, nrows=2, figsize=(15, 10)) # type:ignore
for model_idx, (model_name, model_evals) in enumerate(evals.items()):
    ax = axs.flat[model_idx] # type: ignore
    ax.set_title(model_name) # type: ignore
    show_last_alignments(list(model_evals.values()), ax=ax) # type: ignore
plt.tight_layout() # type: ignore

In [ ]:
fig, axs = plt.subplots(ncols=3, nrows=2, figsize=(15, 8)) # type:ignore
for model_idx, (model_name, model_evals) in enumerate(evals.items()):
    sample_idx = 0
    ax = axs.flat[model_idx] # type: ignore
    ax.set_title(f'{model_name}, sample #{sample_idx}') # type: ignore
    partial_alignments_plot(model_evals[sample_idx], ax=ax) # type: ignore
plt.tight_layout() # type: ignore

In [ ]:
fig, axs = plt.subplots(ncols=3, nrows=2, figsize=(15, 8)) # type:ignore
for model_idx, (model_name, model_evals) in enumerate(evals.items()):
    sample_idx = 0
    ax = axs.flat[model_idx] # type: ignore
    ax.set_title(f'{model_name}, sample #{sample_idx}') # type: ignore
    visualize_history(model_evals[sample_idx].input_chunks, model_evals[sample_idx].output_chunks, ax=ax) # type: ignore
plt.tight_layout() # type: ignore